In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shivanshcoding/rag-dataset-with-evaluation/rbi_notification.pdf
/kaggle/input/datasets/shivanshcoding/rag-dataset-with-evaluation/QAonRBI.json


In [3]:
import os
import time
import shutil
import torch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document

FILE_PATH = "/kaggle/input/datasets/shivanshcoding/rag-dataset-with-evaluation/rbi_notification.pdf"
QA_PATH = "/kaggle/input/datasets/shivanshcoding/rag-dataset-with-evaluation/QAonRBI.json"
VECTOR_DB_PATH = "/kaggle/working/my_chroma_db_"

In [4]:
# from google.colab import userdata
# from google import genai
# from google.genai import types
# from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# EMBEDDING_API_KEY = user_secrets.get_secret("API_KEY_1")
# LLM_API_KEY = user_secrets.get_secret("API_KEY_2")

# # 🤖 Loading Google GenAI Embeddings (using Key #1)
# print("🤖 Loading embedding 2 via Gemini API...")
# embedding_model = GoogleGenerativeAIEmbeddings(
#     model="models/gemini-embedding-2",
#     api_key=EMBEDDING_API_KEY
# )

# # 🤖 Loading Gemini LLM (using Key #2)
# print("🤖 Loading gemini-3.5-flash via Gemini API...")
# llm = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash",
#     temperature=0.1,
#     api_key=LLM_API_KEY
# )

# print("\n✅ Models loaded successfully using isolated API keys!")

In [5]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_core.output_parsers import StrOutputParser

In [14]:
import urllib.request

# 1. System dependencies for Ollama hardware detection
!apt-get update -qq && apt-get install -y -qq pciutils lshw zstd

# 2. Download and install the Ollama Linux binary
!curl -fsSL https://ollama.com/install.sh | sh

print("🚀 Starting Ollama server on the Kaggle GPU...")

# 3. Start the server detached in the background
os.system("nohup ollama serve > ollama.log 2>&1 &")

# 4. Wait for the server to wake up and start listening
for i in range(1, 15):
    try:
        response = urllib.request.urlopen("http://localhost:11434/", timeout=2)
        if response.getcode() == 200:
            print("✅ Ollama is awake and ready!")
            break
    except Exception:
        print(f"⏳ Waiting for server to initialize... (Attempt {i}/15)")
        time.sleep(2)
else:
    print("❌ Ollama failed to start. Check logs.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
🚀 Starting Ollama server on the Kaggle GPU...
⏳ Waiting for server to initialize... (Attempt 1/15)
⏳ Waiting for server to initialize... (Attempt 2/15)
✅ Ollama is awake and ready!


In [15]:
print("📥 Downloading Qwen 27B and embedding 8B (this will take a few minutes)...")
!ollama pull qwen3.6:27b
!ollama pull qwen3-embedding:8b
print("🎉 All done! You are ready to run your RAG pipeline.")

📥 Downloading Qwen 27B and embedding 8B (this will take a few minutes)...
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 83c54730a5fe:   0% ▕                  ▏ 9.7 MB/ 17 GB                  pulling manifest 
pulling 83c54730a5fe:   0% ▕                  ▏  84 MB/ 17 GB                  pulling manifest 
pulling 83c54730a5fe:   1% ▕                  ▏ 124 MB/ 17 GB                  pulling manifest 
pulling 83c54730a5fe:   1% ▕                  ▏ 218 MB/ 17 GB                  pulling manifest 
pulling 83c54730a5fe:   2% ▕                  ▏ 303 MB/ 17 GB                  pulling manifest 
pulling 83c54730a5fe:   2% ▕                  ▏ 348 MB/ 17 GB                  pulling manifest 
pulling 83c54730a5fe:   3% ▕                  ▏ 441 MB/ 17 GB                  pulling manifest 
pulling 83c54730a5fe:   3% ▕                  ▏ 535 MB/ 17 GB        

In [16]:
print("🤖 Loading BGE-M3 for Embeddings...")
embedding_model = OllamaEmbeddings(model="qwen3-embedding:8b")

print("🤖 Loading Qwen 2.5 14B...")
llm = ChatOllama(model="qwen3.6:27b", temperature=0.1)

print("✅ Models loaded successfully into LangChain!")

🤖 Loading BGE-M3 for Embeddings...
🤖 Loading Qwen 2.5 14B...
✅ Models loaded successfully into LangChain!


In [17]:
from langchain_community.document_loaders import PyPDFLoader

print("Loading PDF...")
loader = PyPDFLoader(FILE_PATH)
documents = loader.load()

print("Documents loaded : ", len(documents))

Loading PDF...
Documents loaded :  330


In [18]:
from langchain_experimental.text_splitter import SemanticChunker

def build_semantic_retriever(documents, embedding_fn, threshold=80):
    print("🧹 Cleaning old Semantic DB...")
    shutil.rmtree(f"{VECTOR_DB_PATH}semantic", ignore_errors=True)

    print("✂️ Building Semantic Chunker...")
    semantic_chunker = SemanticChunker(
        embedding_fn,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=threshold
    )

    print("📝 Slicing documents semantically...")
    chunks = semantic_chunker.split_documents(documents)

    print(f"Created {len(chunks)} semantic chunks")

    print("🗄️ Indexing Semantic ChromaDB...")
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_fn,
        persist_directory=f"{VECTOR_DB_PATH}semantic"
    )
    return vector_store.as_retriever(search_kwargs={"k": 20})

In [19]:
def build_parent_child_retriever(documents, embedding_fn):
    print("🧹 Cleaning old Parent-Child DB...")
    shutil.rmtree(f"{VECTOR_DB_PATH}parentchild", ignore_errors=True)

    print("✂️ Setting up Parent and Child splitters...")
    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=250)
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=75)

    vector_store = Chroma(
        collection_name="child_embeddings",
        embedding_function=embedding_fn,
        persist_directory=f"{VECTOR_DB_PATH}parentchild"
    )
    docstore = InMemoryStore()

    print("🔗 Building Parent-Child Retriever...")
    retriever = ParentDocumentRetriever(
        vectorstore=vector_store,
        docstore=docstore,
        child_splitter=child_splitter,
        parent_splitter=parent_splitter,
        search_kwargs={"k": 20}
    )

    print("🗄️ Indexing Parent-Child documents...")
    retriever.add_documents(documents)
    return retriever

In [20]:
print("🚀 INITIALIZING ALL DATABASES...")

pc_retriever = build_parent_child_retriever(documents, embedding_model)
semantic_retriever = build_semantic_retriever(documents, embedding_model)

print("\n✅ All retrievers are loaded into memory and ready to search!")

🚀 INITIALIZING ALL DATABASES...
🧹 Cleaning old Parent-Child DB...
✂️ Setting up Parent and Child splitters...
🔗 Building Parent-Child Retriever...
🗄️ Indexing Parent-Child documents...


KeyboardInterrupt: 

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

print("🔍 Building BM25 Keyword Search...")
keyword_retriever = BM25Retriever.from_documents(documents)
keyword_retriever.k = 10

print("🤝 Fusing Vector and Keyword Search...")
hybrid_retriever = EnsembleRetriever(
    retrievers=[pc_retriever, semantic_retriever, keyword_retriever], 
    weights=[0.4,0.4,0.2]
)

In [ ]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
 
print("🧠 Loading local Cross-Encoder (BAAI BGE-Reranker)...")
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

reranker_compressor = CrossEncoderReranker(model=cross_encoder, top_n=10)

print("🤝 Wrapping Hybrid Retriever with the Reranker...")
advanced_retriever = ContextualCompressionRetriever(
    base_compressor=reranker_compressor,
    base_retriever=hybrid_retriever
)

print("Done")

In [ ]:
rag_template = """
You are a helpful, secure assistant. Answer the user's question based strictly on the provided context.
If you do not know the answer based on the local context, then try to answer from web context, and 
if you dont get the answer from that context too then say "I cannot find that in the context provided."

Answer the user's question using simple, standard,and natural English prose.
STRICT RULES:
- Do NOT use any Markdown formatting whatsoever.
- Do NOT use asterisks (**bold**), hashtags (# headers), lists with dashes, or backticks.
- Write only in plain text paragraphs, like a standard book or email.
- Your answer should answer question properly and upto the mark with the context.

Context:
{context}

Question:
{question}

Answer:
"""
prompt = ChatPromptTemplate.from_template(rag_template)

In [ ]:
def format_docs(docs):
    """Helper function to format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)
rag_chain = (
    {"context": advanced_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
check_template = """
You are a strict evaluator. Compare the 'Reference Answer' and the 'Generated Answer' for semantic similarity.
Focus strictly on factual overlap and core meaning, not exact wording.

Output ONLY a single integer percentage representing the similarity (from 0% to 100%), followed by the '%' sign. Do not write anything else. No explanations.

Question: {question}

Reference Answer: {answer}

Generated Answer: {response}
"""

check_prompt = ChatPromptTemplate.from_template(check_template)
eval_chain = check_prompt | llm | StrOutputParser()

In [ ]:
import json

def create_qa_lookup(filepath):
    print(f"📂 Loading data from {filepath}...")

    with open(filepath, "r", encoding="utf-8") as file:
        qa_list = json.load(file)


    print(f"✅ Successfully fetched {len(qa_list)} Q&A pairs!")
    return qa_list

my_rbi_list = create_qa_lookup(QA_PATH)

In [ ]:
for item in my_rbi_list:
    print(f"Generating answer for question number : {item["question_number"]}")
    response = rag_chain.invoke(item["question"])
    print(f"\nFinal Answer: {response}")
    inputs = {
        "question": item["question"],
        "answer": item["answer"],
        "response": response
    }
    eval_result = eval_chain.invoke(inputs)
    print("\nThe real answer matches the RAG answer by:",eval_result)
    print("-----------------------------------------------------------------------------------------")

In [ ]:
my_list = [{"question":"Who is Shivansh Rana?","answer":"Shivansh Rana is SDE Intern at Hero Motocorp, Bangalore"},
          {"question":"What is full form of RBI?","answer":"Reserve Bank of India"}]

for item in my_list:
    print(f"Generating answer for question  : {item["question"]}")
    response = rag_chain.invoke(item["question"])
    print(f"\nFinal Answer: {response}")
    inputs = {
        "question": item["question"],
        "answer": item["answer"],
        "response": response
    }
    eval_result = eval_chain.invoke(inputs)
    print("\nThe real answer matches the RAG answer by:",eval_result)
    print("-----------------------------------------------------------------------------------------")